In [ ]:
import os, sys, torch, torch.nn.functional as F
import pandas as pd, re, random
from sklearn.model_selection import train_test_split
from modelos.cvcnn import ComplexConv1DNet
sys.path.insert(0, r"c:\repos\DroneDetectionRF\NoisyUAV")
from funciones.cargador import cargar_muestra, NOMBRES_CLASES
from funciones.detector_entropia import detectar_bursts, plot_muestra, print_diagnostico, plot_espectrograma_3d

In [ ]:
TARGET_DESEADO = 3      # 0=DJI, 1=FutabaT14, 2=FutabaT7, 3=Graupner,
                         # 5=Taranis, 6=Turnigy  |  None = cualquier dron
SNR_DESEADA    = -12     # dB, paso 2 desde -20 hasta 30  |  None = cualquiera

In [ ]:
# 1. EMPAREJAMIENTO ESTRICTO CON LA DIVISIÓN DE TEST
CSV_METADATA = r"C:\TFM_data\NoisyUAV\stage2_norm\metadata_stage2.csv"
if not os.path.exists(CSV_METADATA):
    CSV_METADATA = r"C:\TFM_data\NoisyUAV\stage2\metadata_stage2.csv"

In [ ]:
df = pd.read_csv(CSV_METADATA)
df_train, df_temp = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=42)
df_val,   df_test = train_test_split(df_temp, test_size=0.50, stratify=df_temp["label"], random_state=42)

In [ ]:
candidatos = df_test[df_test["label"] == 1].copy()   # solo drones
if TARGET_DESEADO is not None:
    candidatos = candidatos[candidatos["target"] == TARGET_DESEADO]
    if candidatos.empty:
        raise ValueError(f"❌ No hay muestras de target={TARGET_DESEADO} en el test set.")
if SNR_DESEADA is not None:
    candidatos = candidatos[candidatos["snr"] == SNR_DESEADA]
    if candidatos.empty:
        snrs_disponibles = sorted(df_test[df_test["label"] == 1]["snr"].unique())
        raise ValueError(
            f"❌ No hay muestras de target={TARGET_DESEADO} con SNR={SNR_DESEADA} dB.\n"
            f"   SNRs disponibles para este target: {snrs_disponibles}"
        )

In [ ]:
nombre_clase = NOMBRES_CLASES.get(TARGET_DESEADO, f"target{TARGET_DESEADO}") if TARGET_DESEADO is not None else "cualquier dron"
print(f"✅ {len(candidatos)} candidatos encontrados → [{nombre_clase} | SNR={SNR_DESEADA} dB]")

In [ ]:
# semilla_aleatoria = random.randint(0, 9999)
# Para prueba con un dato en particular
semilla_aleatoria = 1318
sample_seleccionado = candidatos.sample(1, random_state=semilla_aleatoria).iloc[0]
crop_id        = sample_seleccionado["crop_id"]

# Prueba con un fichero específico
# parent_filename = re.sub(r"_burst\d+\.pt$", ".pt", crop_id)
parent_filename = "IQdata_sample10316_target4_snr26.pt"

RUTA_RAW_PARENT = os.path.join(r"C:\TFM_data\NoisyUAV\drone_RF_data", parent_filename)

print("=" * 52)
print(f"🎬 Muestra Ciega Recuperada    : {parent_filename}, correspondiente con la semilla {semilla_aleatoria}")
print(f"📡 Target : {sample_seleccionado['target']} ({nombre_clase})"
      f"  |  SNR : {sample_seleccionado['snr']} dB")
print("=" * 52)

In [ ]:
iq_tensor, _, original_target, original_snr = cargar_muestra(RUTA_RAW_PARENT)
print(f"✅ Tensor cargado: {iq_tensor.shape}")

In [ ]:
# Variables Físicas de la Tesis
FS = 14e6
NPERSEG = 2048
Z_THRESH = 2.0      
MIN_BURST_MS = 0.5
MERGE_GAP_MS = 2.5
MIN_Z_ABS = 2.5
BG_MULT = 4
MAX_BINS_FRAC = 0.25
SMOOTH_MS = 0.2
ADAPTIVE_WINDOW_MS = 10 

# Detección
t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts = detectar_bursts(
    iq_tensor, fs=FS, nperseg=NPERSEG, z_thresh=Z_THRESH,
    min_burst_ms=MIN_BURST_MS, merge_gap_ms=MERGE_GAP_MS,
    min_z_abs=MIN_Z_ABS, bg_mult=BG_MULT, max_bins_frac=MAX_BINS_FRAC,
    smooth_ms=SMOOTH_MS, adaptive_window_ms=ADAPTIVE_WINDOW_MS,
)

print_diagnostico(
    t_ms, nf_v, ns, umbral_v, n_active, bursts,
    nperseg=NPERSEG, fs=FS, z_thresh=Z_THRESH,
    bg_mult=BG_MULT, max_bins_frac=MAX_BINS_FRAC,
    min_burst_ms=MIN_BURST_MS, merge_gap_ms=MERGE_GAP_MS,
    target=f"Target {original_target}", snr=f"{original_snr}", index=0,
)

# Magia Visual de tu Proyecto
fig_2d = plot_muestra(
    iq_tensor, t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts,
    fs=FS, nperseg=NPERSEG, z_thresh=Z_THRESH,
    bg_mult=BG_MULT, max_bins_frac=MAX_BINS_FRAC,
    adaptive_window_ms=ADAPTIVE_WINDOW_MS,
    titulo=f"Validación Test End-To-End: {parent_filename}"
)
fig_2d.show()

# fig_3d = plot_espectrograma_3d(
#     iq_tensor, fs=FS, nperseg=NPERSEG, 
#     t_lim_ms=None, smooth_sigma=2.0, floor_pct=5,
#     titulo="Espectro Base (NoisyUAV 14 MHz)"
# )
# fig_3d.show()


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ComplexConv1DNet(num_classes=2, pool_output_size=64, dropout=0.5)

# IMPORTANTE: Cargamos el Universal (Normalized) porque ese es el bueno.
ruta_pesos = r"C:\TFM_data\NoisyUAV\stage2_norm\checkpoints\cvcnn_best_model.pth"
# ruta_pesos = r"C:\TFM_data\NoisyUAV\stage2\checkpoints\cvcnn_best_model.pth"
model.load_state_dict(torch.load(ruta_pesos, map_location=device, weights_only=True))
model.to(device)
model.eval()

# ===== 1. CROP FÍSICO Y MEMORIA =====
muestras_recortadas = []
for b in bursts:
    idx_inicio = int(b['t0'] * 1e-3 * FS)
    idx_fin = int(b['t1'] * 1e-3 * FS)
    pulso = iq_tensor[:, idx_inicio:idx_fin]
    muestras_recortadas.append(pulso)

print(f"\n✅ Acabamos de aislar {len(muestras_recortadas)} micro-ráfagas FHSS del CFAR.")
print("==============================================")
print("  VEREDICTO CV-CNN (NORMALIZADA A TIEMPO REAL) ")
print("==============================================")

drones_encontrados = 0
ruidos_encontrados = 0

with torch.no_grad():
    for i, pulso in enumerate(muestras_recortadas):
        
        # OBLIGATORIO: NORMALIZAR CADA PULSO (AGC)
        pulso_normalizado = pulso / torch.max(torch.abs(pulso))
        input_ia = pulso_normalizado.unsqueeze(0).to(device) 
        
        outputs = model(input_ia)
        prob = F.softmax(outputs, dim=1)
        _, prediccion = outputs.max(1)
        prob_dron = prob[0][1].item() * 100 

        t_ms_inicio = bursts[i]['t0']
        
        if prediccion.item() == 1:
            drones_encontrados += 1
            print(f"  [B{i+1:02d}] t={t_ms_inicio:6.2f} ms | 🧠 IA: DRON   [{prob_dron:6.2f}%] ✅")
        else:
            ruidos_encontrados += 1
            print(f"  [B{i+1:02d}] t={t_ms_inicio:6.2f} ms | 🧠 IA: RUIDO  [{prob_dron:6.2f}%] ❌")

print("----------------------------------------------")
print(f"Resumen Detecciones: {drones_encontrados} Drones | {ruidos_encontrados} Falsas Alarmas")
